# Project Starter: Option 5, Your Own Approved Data

**Data:** a dataset you chose and the instructor approved, such as public reviews or survey answers that you download yourself. Until your data is in place, the notebook runs on the course complaint file as a **demo**, so you can see every step work.

This notebook sets up your project data the same way every time. It loads the data, records where the data came from, audits it, protects the test data and saves the outputs that Milestone 1 (MS1) asks for. It does not build your final method.

| Part | What it does |
|---|---|
| 1 Setup | Packages, folders and your settings: file, columns and split method |
| 2 Load and provenance | Load your file and record where it came from |
| 3 Personal details and IDs | Drop personal columns and unusable texts, scan the text, check or build stable IDs |
| 4 Audit | Short and repeated text, length, label balance, dates, reading |
| 5 Protected split | Make a random, time or group split once, save it and add a guard; for your own label, samples to label and how to load them |
| 6 Evaluation plan | Write down the plan before you see any results |
| 7 First baseline (for MS2) | A default guess and a simple model; off until MS2 |
| 8 Save | Save the outputs and the reproducibility record, then commit the safe files only |

**How to use it.**

1. Copy this file into your project folder, `project/notebooks/`, and give it a clear name, such as `01_data_audit_and_split.ipynb`.
2. Fill in the settings in Part 1. You can run the notebook first with the settings as they are: they point to the course complaint file as a demo.
3. Run all cells from the top (**Restart**, then **Run All**). Read each output before you go on. Rerunning is always safe: the notebook never overwrites your label files.
4. Cells marked **Your turn** are for your own work, with a notes cell below each. The notebook runs without them.
5. Commit the executed notebook with MS1. The MS1 brief lists everything else your repository must hold.

This starter has no points. The milestone rubrics in Canvas describe how the project is graded.

## Part 1 Setup

### 1.1 Packages

Select the `ban6403` kernel, then run the cell. The tools for the reproducibility record (hashes and package versions) are loaded at the end, in the last part.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import textwrap
import unicodedata
import pandas as pd

### 1.2 The folders

Run this notebook from `project/notebooks/`. The repository is then two folders up, `../..`, and every path below starts from there. This notebook writes to two places, and the cell makes them if they do not exist yet:

- `project/outputs/option5_setup/`: audit tables, the split and the ID lists (IDs only, no text), the evaluation plan and the manifest. These are safe to commit.
- `project/data/private/`: anything that contains text from your data. Keep your downloaded raw file there too; the label files go to `project/data/private/annotations/`. The repository's `.gitignore` keeps this folder out of Git. If your written approval allows you to commit a sample, you may set `ANNOT = REPO / "project/data/annotations"` instead, so the instructor can check your labels.

In [ ]:
REPO = Path("../..")   # this notebook runs from project/notebooks/, two folders below the repository
OUT = REPO / "project/outputs/option5_setup"
PRIVATE = REPO / "project/data/private"
ANNOT = PRIVATE / "annotations"
# mkdir makes a folder; parents=True also makes missing parent folders; exist_ok=True skips a folder that exists.
for folder in [OUT, PRIVATE, ANNOT]:
    folder.mkdir(parents=True, exist_ok=True)

### 1.3 Your settings

Describe your file in the settings cell. Column names must match your file exactly.

| Setting | What to enter |
|---|---|
| `DATA_PATH` | Path to your raw file, relative to the repository, such as `"project/data/private/my_reviews.csv"` |
| `ID_COL` | The column with a stable record ID, or `""` if your file has none (the notebook then builds one). The IDs must be unique and never empty. |
| `TEXT_COL` | The column with the text |
| `LABEL_COL` | The column you will predict or describe, or `"own_label"` if you will label your own target (Part 5.3) |
| `DATE_COL` | A date column, or `""` |
| `GROUP_COL` | A column whose records must stay together, such as a product ID, or `""` |
| `DROP_COLUMNS` | Columns with personal details to remove at once, such as reviewer names |
| `ID_PARTS` | Columns that together identify a record, used only when `ID_COL` is empty |
| `SPLIT_METHOD` | `"random"`, `"time"` (newest records become test) or `"group"` (whole groups go to one split) |
| `APPROVED_ON` | The date the instructor approved your dataset |
| `RUN_MS2_BASELINE` | Keep `False` for MS1. Set it to `True` for MS2; Part 7 then fits a first model. |

**Example for a review file without review IDs.** Suppose your file has the columns `product` (the item reviewed), `user_name` (the reviewer), `rating`, `date`, `title` and `body`. Check the column names in your own download:

```python
DATA_PATH = "project/data/private/my_reviews.csv"
ID_COL = ""
TEXT_COL = "body"
LABEL_COL = "rating"
DATE_COL = "date"
GROUP_COL = "product"
DROP_COLUMNS = ["user_name"]
ID_PARTS = ["product", "date", "title", "body"]
SPLIT_METHOD = "group"
```

In [ ]:
# Demo settings: the course complaint file. Replace them with your own.
DATA_PATH = "data/course/cfpb_complaints.csv"
ID_COL = "complaint_id"
TEXT_COL = "narrative"
LABEL_COL = "product"     # a column of your data, or "own_label" for your own label (Part 5.3)
DATE_COL = "date_received"
GROUP_COL = ""
DROP_COLUMNS = ["state"]
ID_PARTS = []
SPLIT_METHOD = "time"
APPROVED_ON = ""
DECISION_OWNER = ""
BUSINESS_QUESTION = ""
RUN_MS2_BASELINE = False  # True from MS2 on: Part 7 then fits a first model
SEED = 6403               # fixes every random draw, so each run gives the same samples

# DEMO is True while DATA_PATH is still the course file; OWN_LABEL is True when you label your own target.
DEMO = DATA_PATH == "data/course/cfpb_complaints.csv"
OWN_LABEL = LABEL_COL == "own_label"
SETTINGS = {"data_path": DATA_PATH, "id_col": ID_COL, "text_col": TEXT_COL, "label_col": LABEL_COL,
            "date_col": DATE_COL, "group_col": GROUP_COL, "drop_columns": DROP_COLUMNS, "id_parts": ID_PARTS,
            "split_method": SPLIT_METHOD, "approved_on": APPROVED_ON,
            "decision_owner": DECISION_OWNER, "business_question": BUSINESS_QUESTION,
            "run_ms2_baseline": RUN_MS2_BASELINE}
if DEMO:
    print("DEMO MODE: the notebook uses the course complaint file. Replace DATA_PATH and the columns with your own data.")
# These three settings must be filled in before MS1.
for name in ["approved_on", "decision_owner", "business_question"]:
    if not SETTINGS[name]:
        print(f"CHECK: {name} is empty")

## Part 2 Load the data and record its provenance

### 2.1 Load the file

The cell reads your ID column as text, so IDs such as `00123` keep their leading zeros. The SHA-256 hash of the file goes into the reproducibility record at the end (Part 8). Copy it into your README: anyone who downloads the data again can check that they have the same version.

In [ ]:
DATA_FILE = REPO / DATA_PATH
# Read the ID column as text (dtype maps a column name to a type).
id_types = {}
if ID_COL:
    id_types[ID_COL] = str
# low_memory=False reads the whole file before it chooses the column types (fewer type warnings).
raw = pd.read_csv(DATA_FILE, dtype=id_types, low_memory=False)
print("rows:", len(raw), "| columns:", list(raw.columns))

### 2.2 Provenance record

Fill in the record for your data. If you set `DATE_COL`, the cell fills the time window from the dates, read as dates (sorting them as text would put "April" before "January"). It goes into your data dictionary and into the approval form. In demo mode it describes the course file.

In [ ]:
provenance = {
    "source": "",           # who published the data, and where you downloaded it (link)
    "downloaded_on": "",    # the date you downloaded it
    "license_or_terms": "", # the license or terms of use, in your own words
    "collection": "",       # how the texts were collected (for example scraped reviews, survey answers)
    "time_window": "",      # the period the texts cover
    "unit_of_analysis": "", # what one row is (one review, one ticket ...)
    "personal_details": "", # what personal details exist, and how you removed them
    "may_be_shared": "",    # yes / no: may the raw file or a sample of it be committed to your private course repository?
}
if DEMO:
    provenance.update({"source": "CFPB complaint narrative archive (course extract)", "license_or_terms": "public data",
                       "collection": "consumer complaints submitted to CFPB", "unit_of_analysis": "one complaint",
                       "personal_details": "redacted by CFPB; the demo also drops the state column", "may_be_shared": "yes"})

def parse_dates(values):
    """Take a column of dates written as text; return them as dates (unreadable ones become NaT, missing)."""
    # errors="coerce" turns an unreadable date into NaT; format="mixed" allows several date formats.
    return pd.to_datetime(values, errors="coerce", format="mixed")

# Fill in the time window from the dates, unless you wrote it yourself.
if DATE_COL and not provenance["time_window"]:
    raw_dates = parse_dates(raw[DATE_COL])
    if raw_dates.notna().any():
        provenance["time_window"] = (f"{raw_dates.min().date()} to {raw_dates.max().date()} "
                                     f"({int(raw_dates.notna().sum()):,} of {len(raw):,} dates readable)")
provenance["rows"] = len(raw)

# Print each entry; an empty one is marked CHECK.
for key, value in provenance.items():
    if value in ("", None):
        value = "CHECK: fill in"
    print(f"{key:>18}: {value}")

## Part 3 Personal details and stable IDs

### 3.1 Drop personal columns and unusable texts

Remove columns with personal details (names, user handles, emails, locations you do not need) as the first step. Later steps never see them. The cell then leaves out records without a usable text. It counts two cases apart: a **missing** text (an empty cell) and a **blank** text (only spaces). Report both counts in your data section.

In [ ]:
data = raw.drop(columns=DROP_COLUMNS)
print("dropped columns:", DROP_COLUMNS)
# isna() is True for an empty cell; str.strip().eq("") is True for a text of only spaces.
missing_text = data[TEXT_COL].isna()
blank_text = data[TEXT_COL].fillna("x").str.strip().eq("")
exclusions = pd.DataFrame([
    {"reason": "text missing (empty cell)", "records": int(missing_text.sum()), "of": len(raw)},
    {"reason": "text blank (only spaces)", "records": int(blank_text.sum()), "of": len(raw)},
])
# | means "or" and ~ means "not": keep the records with a usable text.
data = data[~(missing_text | blank_text)].copy()
# A copy of your text column under a fixed name, used by every later step.
data["text"] = data[TEXT_COL].astype(str)
print(exclusions.to_string(index=False))
print("records with a usable text:", len(data), "of", len(raw))

### 3.2 Scan the text for personal details

People write names, phone numbers and emails into free text. The scan counts texts that match common patterns. A match is only a lead: read some of them, and decide whether to remove the detail, mask it or drop the record. Write the decision in your data dictionary.

In [ ]:
# Patterns that may be personal details. A match is a reason to read the text, not proof.
# Pattern key: \w a letter or digit, \d a digit, \s a space, \b a word edge, {3} exactly three, + one or more.
PII_PATTERNS = {
    "email address": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone number": r"\(?\b\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}\b",
    "web link": r"https?://\S+|www\.\S+",
    "long digit string (card or account?)": r"\b\d{9,}\b",
    "@handle": r"(?<![\w.])@\w{3,}",
}

def pii_scan(texts):
    """Take a column of texts; return how many texts match each pattern above."""
    rows = []
    for name, pattern in PII_PATTERNS.items():
        # True for each text with at least one match.
        hits = texts.str.contains(pattern, regex=True)
        rows.append({"pattern": name, "texts with a match": int(hits.sum()), "of": len(texts)})
    return pd.DataFrame(rows)

In [ ]:
pii_table = pii_scan(data["text"])
pii_table

### 3.3 Stable IDs

Every output of your project must link back to one record, so every record needs its own ID that never changes. If you set `ID_COL`, the cell checks that the IDs are unique and never empty, and stops if they are not. Then choose another ID column, or set `ID_COL = ""`. With `ID_COL = ""`, the cell builds an ID from the columns in `ID_PARTS`: a short hash of their values, plus a counter when two records are identical. The same file always gives the same IDs.

In [ ]:
if ID_COL:
    ids = data[ID_COL]
    n_missing = int(ids.isna().sum())
    n_repeated = int(ids.dropna().duplicated().sum())
    if n_missing or n_repeated:
        raise ValueError(f"{ID_COL} has {n_missing} empty and {n_repeated} repeated values; every record needs its own ID. "
                         'Choose another ID column, or set ID_COL = "" to build IDs from ID_PARTS.')
    data["record_id"] = ids
else:
    # No ID column: build an ID from a hash of some columns (the text when ID_PARTS is empty).
    if ID_PARTS:
        parts = ID_PARTS
    else:
        parts = [TEXT_COL]
    # agg("|".join, axis=1) joins the chosen columns of each row into one text, separated by "|".
    key_texts = data[parts].fillna("").astype(str).agg("|".join, axis=1)
    hashes = []
    for value in key_texts:
        # sha256 gives a 64-character code; [:12] keeps the first 12, enough to tell records apart.
        hashes.append(hashlib.sha256(value.encode()).hexdigest()[:12])
    base = pd.Series(hashes, index=data.index)
    # cumcount numbers the copies of a repeated hash 0, 1, 2, ..., so every ID is unique.
    data["record_id"] = base + "-" + base.groupby(base).cumcount().astype(str)
print("records:", len(data), "| unique IDs:", data["record_id"].nunique())
data[["record_id"]].head(3)

## Part 4 Audit the corpus

### 4.1 Short and repeated text

The **text key** ignores case, punctuation and extra spaces, in any alphabet, so near-copies count as copies. Accented and non-Latin letters count as letters. A text of only symbols or emoji has an empty key; the cell gives each such text its own key and counts them, so read a few. The column `unit` says what each row counts, and `of` gives the total of that unit.

In [ ]:
def text_key(text):
    """Take one text; return a key that ignores case, punctuation and extra spaces, in any alphabet."""
    # NFKC writes look-alike characters the same way; casefold() is lower() for every alphabet.
    text = unicodedata.normalize("NFKC", text).casefold()
    # \W+ is a run of characters that are not letters or digits; accented and non-Latin letters count as letters.
    return re.sub(r"\W+", " ", text).strip()

# map applies text_key to every text in the column.
data["text_key"] = data["text"].map(text_key)
# A text of only symbols or emoji has an empty key. Give it its own key, so such texts do not count as copies.
empty_key = data["text_key"].eq("")
data.loc[empty_key, "text_key"] = "empty-" + data.loc[empty_key, "record_id"]

# str.split().str.len() counts the words of each text.
data["n_words"] = data["text"].str.split().str.len()
n_records = len(data)
audit = pd.DataFrame([
    {"check": "text missing or blank (left out in 3.1)", "count": int(exclusions["records"].sum()), "unit": "rows in the file", "of": len(raw)},
    {"check": "fewer than 5 words", "count": int(data["n_words"].lt(5).sum()), "unit": "records", "of": n_records},
    {"check": "text of only symbols or emoji", "count": int(empty_key.sum()), "unit": "records", "of": n_records},
    {"check": "repeated text: copies beyond the first", "count": int(data["text_key"].duplicated().sum()), "unit": "records", "of": n_records},
])
print(audit.to_string(index=False))
print()
# describe() gives the quartiles (25%, 50% = median, 75%) and more; keep five of them.
print(data["n_words"].describe()[["min", "25%", "50%", "75%", "max"]].to_string())

### 4.2 Labels and dates

Check the balance of your label and the period your data covers. A label with very few records cannot be evaluated well. Records without a label are counted; Part 7 leaves them out.

In [ ]:
label_counts = None
if OWN_LABEL:
    print("LABEL_COL is your own label: Part 5.5 counts your labels.")
elif LABEL_COL in data.columns:
    print("records without a label:", int(data[LABEL_COL].isna().sum()))
    label_counts = data[LABEL_COL].value_counts()
    print("label values:", len(label_counts))
    print(label_counts.head(15).to_string())
else:
    print(f'CHECK: LABEL_COL "{LABEL_COL}" is not a column. Use a column of your data, or "own_label".')
if DATE_COL:
    data["_date"] = parse_dates(data[DATE_COL])
    print("\ndates read:", int(data["_date"].notna().sum()), "of", len(data),
          "| from", data["_date"].min().date(), "to", data["_date"].max().date())

### 4.3 Is the data big enough?

There is no fixed minimum. What matters is whether your test set can show a difference that matters for the decision. The note flags labels with fewer than 30 records in a 20% test set.

In [ ]:
size_check = pd.DataFrame({"records": [len(data)], "expected test records (20%)": [int(0.2 * len(data))]})
if label_counts is not None:
    # A label with fewer than 30 expected test records (20% of its records) gives an unstable test score.
    small = label_counts[label_counts * 0.2 < 30]
    print("labels with fewer than 30 expected test records:", len(small), "of", len(label_counts))
size_check

## Part 5 Make and protect the split

You make the split **once**, save it, and never redraw it. Later runs read the saved file.

- `"random"`: 60% of distinct texts go to train, 20% to valid and 20% to test, at random with your seed. Copies of one text always share a split. If `LABEL_COL` is a column, each label is split this way on its own (a **stratified** split), so every split keeps about the same label shares. If a label has fewer than 5 distinct texts, the cell says so and splits without the label.
- `"time"`: the cell first finds two cutoff dates: the dates 60% and 80% of the way through your records. Records before the first cutoff go to train, records between the cutoffs to valid, and the newest to test. Use it when your decision is about future texts. Every record needs a readable date.
- `"group"`: 60% of groups (for example phone models) go to train, 20% to valid and 20% to test. The test set then has no group the model saw in training.

After the split, a repeated text stays only in its earliest split, in the order `train`, `valid`, `test`. With `"time"`, this drops a later copy of an earlier text, so the test set stays truly newer. The dropped IDs go to `dropped_cross_split_copies.csv`. The table at the end shows the size and, with dates, the period of each split. Check that the label shares look similar across splits.

### 5.1 Make the split

In [ ]:
if SPLIT_METHOD == "time" and data["_date"].isna().any():
    # Records without a date cannot be placed in time.
    raise ValueError("Some records have a missing or unreadable date. Fix or drop them, or choose 'random' or 'group'.")

def make_split(frame):
    """Take the records; return a column with train, valid or test for each record, following SPLIT_METHOD."""
    if SPLIT_METHOD == "time":
        # Cutoff dates first: the dates 60% and 80% of the way through the records.
        valid_start = frame["_date"].quantile(0.6)
        test_start = frame["_date"].quantile(0.8)
        split = pd.Series("train", index=frame.index)
        split.loc[frame["_date"].ge(valid_start)] = "valid"
        split.loc[frame["_date"].ge(test_start)] = "test"
        return split
    # The units to shuffle: whole groups, or distinct texts (so copies of one text share a split).
    if SPLIT_METHOD == "group":
        units = frame[GROUP_COL].fillna("missing group").astype(str)
    else:
        units = frame["text_key"]
    # One list of units to shuffle, or, for a random split with a label column, one list per label.
    unit_lists = [sorted(units.unique())]
    if SPLIT_METHOD == "random" and LABEL_COL in frame.columns:
        # Stratified split: each label is split 60/20/20 on its own, so every split keeps about the same
        # label shares. A text counts under the label of its first record.
        unit_label = frame[LABEL_COL].fillna("missing label").astype(str).groupby(units).first()
        if unit_label.value_counts().min() >= 5:
            unit_lists = [sorted(unit_label.index[unit_label.eq(label)]) for label in sorted(unit_label.unique())]
        else:
            print("A label has fewer than 5 distinct texts, so the split is random but not stratified by label.")
    elif SPLIT_METHOD == "random":
        print("No label column, so the split is random but not stratified by label.")
    unit_split = {}
    for unit_list in unit_lists:
        # Shuffle the units with the seed; sorted() above makes the order independent of the file order.
        order = pd.Series(unit_list).sample(frac=1, random_state=SEED).tolist()
        n_units = len(order)
        # The first 60% of units go to train, the next 20% to valid, the rest to test.
        for i, unit in enumerate(order):
            if i < round(0.6 * n_units):
                unit_split[unit] = "train"
            elif i < round(0.8 * n_units):
                unit_split[unit] = "valid"
            else:
                unit_split[unit] = "test"
    # map looks up each record's unit in the dictionary.
    return units.map(unit_split)

# The demo split and your own split are kept in different files.
if DEMO:
    SPLIT_FILE = OUT / "demo_split.csv"
else:
    SPLIT_FILE = OUT / "project_split.csv"
# Some downloads already have a "split" column; drop it (errors="ignore": no error if there is none).
data = data.drop(columns="split", errors="ignore")
if SPLIT_FILE.exists() and not DEMO:
    saved_split = pd.read_csv(SPLIT_FILE, dtype={"record_id": str})
    # The saved split must cover exactly the records of this run.
    if set(saved_split["record_id"]) != set(data["record_id"]):
        raise ValueError("The records differ from the saved split. Restore your data file and settings, "
                         "or ask the instructor before you make a new split.")
    # merge brings in the saved split by ID; the IDs are unique, so each record gets exactly one split.
    data = data.merge(saved_split, on="record_id", how="left")
    print("Loaded the saved split:", SPLIT_FILE.name)
else:
    data["split"] = make_split(data)
    data[["record_id", "split"]].to_csv(SPLIT_FILE, index=False)
    print("Made and saved a new split with method", SPLIT_METHOD, "->", SPLIT_FILE.name)

# The text keys found in train, and those found in valid.
train_keys = set(data.loc[data["split"].eq("train"), "text_key"])
valid_keys = set(data.loc[data["split"].eq("valid"), "text_key"])
# isin is True where a key is in the set; & means "and", | means "or".
valid_copy = data["split"].eq("valid") & data["text_key"].isin(train_keys)
test_copy = data["split"].eq("test") & (data["text_key"].isin(train_keys) | data["text_key"].isin(valid_keys))
is_copy = valid_copy | test_copy
dropped_copies = data.loc[is_copy, ["record_id", "split"]]
dropped_copies.to_csv(OUT / "dropped_cross_split_copies.csv", index=False)
# ~ means "not": keep the records that are not dropped copies.
data = data[~is_copy].copy()
print("copies dropped from valid:", int(valid_copy.sum()), "| from test:", int(test_copy.sum()),
      "| records left:", len(data))

# Records per split, and with a date column, the first and last date of each split.
split_table = data.groupby("split").size().rename("records").to_frame()
if DATE_COL:
    split_table["first date"] = data.groupby("split")["_date"].min().dt.date
    split_table["last date"] = data.groupby("split")["_date"].max().dt.date
split_table = split_table.reindex(["train", "valid", "test"])
split_table

Outside demo mode, the cell loads `project_split.csv` if it exists, instead of making a new split. It stops if the records in your file are not exactly the records in the saved split. The demo writes `demo_split.csv`, which your own run never reads. Delete `project_split.csv` only if the instructor agrees that the split must change, and record why.

### 5.2 Record the test IDs and a guard

The test IDs go to `project/outputs/option5_setup/protected_test_ids.csv`; their fingerprint goes into the manifest at the end. The guard `check_protected` stops a cell if a table you are about to fit or tune on holds a protected record. Protected means a test record, and, if you make your own label, a record of your held-out sample. Call the guard before every fit until MS3.

In [ ]:
test_ids = data.loc[data["split"].eq("test"), "record_id"].tolist()
# Save the test IDs, so you can show later that no method used them before MS3.
pd.DataFrame({"record_id": sorted(test_ids)}).to_csv(OUT / "protected_test_ids.csv", index=False)
# A set of protected IDs; the held-out sample is added to it if you make your own label.
protected_ids = set(test_ids)

def check_protected(frame, id_col="record_id"):
    """Take a table; stop with an error if any of its IDs is protected (test, or your held-out sample)."""
    # & keeps the IDs that are in both sets.
    overlap = set(frame[id_col]) & protected_ids
    if overlap:
        raise ValueError(f"{len(overlap)} protected records found. Do not fit or tune on them before MS3.")
    print(f"ok: no protected records among {len(frame):,} rows")

print("protected test records:", len(test_ids))
check_protected(data[data["split"].ne("test")])

if GROUP_COL:
    # Groups with records in both train and test (should be 0 with the group method).
    train_groups = set(data.loc[data["split"].eq("train"), GROUP_COL])
    test_groups = set(data.loc[data["split"].eq("test"), GROUP_COL])
    print("groups in both train and test:", len(train_groups & test_groups))

### 5.3 Read before you count

Numbers do not replace reading. The helper prints a few **training** records, drawn with your seed. Read only `train` records: `valid` may hold your held-out sample, and `test` is your final check. The preview cuts each text, which is fine for browsing. When you judge a label, read the whole text, because a later sentence can change it: use `chars=None`.

In [ ]:
def show_examples(frame, n=3, seed=SEED, chars=300):
    """Take a table; print n records drawn with the seed, each cut to chars characters (chars=None: full text)."""
    examples = frame.sample(min(n, len(frame)), random_state=seed)
    # itertuples gives one row at a time; row.text is the value in the text column.
    for row in examples.itertuples(index=False):
        print(row.record_id)
        shown = row.text
        if chars is not None and len(shown) > chars:
            shown = shown[:chars] + "..."
        # textwrap.fill breaks the text into lines of at most 100 characters.
        print(textwrap.fill(shown, 100))
        print()

# Training records only; valid and test stay unread.
train_records = data[data["split"].eq("train")]
show_examples(train_records)

**Your turn A.** Read at least 15 training records in full, drawn with your seed. Note any personal details the scan missed, what you will do about them, and two cases that would be hard to label. If your data may not be shared, clear the outputs of every cell that prints text before you save the notebook (right-click an output, then **Clear Cell Outputs**), because the executed notebook is committed.

In [ ]:
# Your turn A
# TODO 1: print 15 training records in full with another seed: show_examples(train_records, n=15, seed=1, chars=None)
# TODO 2: write the IDs, your privacy decisions and two hard cases in the cell below.

**Your notes (Your turn A)**

- Records read (IDs):
- Personal details the scan missed, and your decision (remove, mask or drop):
- Hard case 1 (ID, why it is hard):
- Hard case 2 (ID, why it is hard):

### 5.4 Samples to label (only for your own label)

Many projects need a label that the data does not have, such as "mentions a delivery problem". Then set `LABEL_COL = "own_label"` in Part 1.3, and read and label records yourself with your annotation guide. Every own-label project follows the same route:

- The **development sample** comes from `train`. Use it to write and test your guide, and to build and tune your methods.
- The **held-out sample** comes from `valid`. It is your evaluation set for MS3. Label it with the finished guide by the MS2 freeze, before any method sees it. With your own label, `valid` is not used for tuning, and the guard blocks the held-out records.
- Neither sample comes from `test`, so you never read a test record before MS3.

**How the files work.** The cell writes two blank templates to `project/data/private/annotations/`: `dev_labels_template.csv` and `heldout_labels_template.csv`. Copy each one to `dev_labels.csv` and `heldout_labels.csv`, and fill the `own_label` column in the copies. The notebook only reads these two files, so a rerun never erases your labels.

A template is written only when neither it nor your copy exists. To get a fresh template, for example after you change a sample size, delete the old template first. Do not use `NA` or `None` as a label value: pandas reads them as empty. The samples contain text, so they stay in the private folder.

In [ ]:
N_DEV_LABEL = 100
N_HELDOUT_LABEL = 100
DEV_TEMPLATE = ANNOT / "dev_labels_template.csv"
HELDOUT_TEMPLATE = ANNOT / "heldout_labels_template.csv"
DEV_LABELS = ANNOT / "dev_labels.csv"           # your labeled copy: the notebook only reads it
HELDOUT_LABELS = ANNOT / "heldout_labels.csv"   # your labeled copy: the notebook only reads it

def label_sample(split, n):
    """Take a split name and a size; return a seeded sample of that split with empty own_label and note columns."""
    pool = data[data["split"].eq(split)]
    # min(n, len(pool)) never asks for more rows than the split has; random_state=SEED fixes the draw.
    sample = pool.sample(min(n, len(pool)), random_state=SEED)
    sample = sample[["record_id", "text"]].sort_values("record_id")
    sample["own_label"] = ""
    sample["note"] = ""
    return sample

def write_template(sample, template, completed):
    """Save a blank template, but only if neither the template nor your completed file exists yet."""
    if template.exists() or completed.exists():
        print("kept as it is:", template.name, "and", completed.name)
    else:
        sample.to_csv(template, index=False)
        print("wrote", template.name)

heldout_ids = []
if OWN_LABEL:
    dev_sample = label_sample("train", N_DEV_LABEL)
    heldout_sample = label_sample("valid", N_HELDOUT_LABEL)
    write_template(dev_sample, DEV_TEMPLATE, DEV_LABELS)
    write_template(heldout_sample, HELDOUT_TEMPLATE, HELDOUT_LABELS)
    heldout_ids = heldout_sample["record_id"].tolist()
    # From now on the guard also blocks the held-out records.
    protected_ids.update(heldout_ids)
    print("development sample (train):", len(dev_sample), "| held-out sample (valid):", len(heldout_ids))
else:
    print(f"LABEL_COL is the column '{LABEL_COL}', so you do not need samples to label.")

### 5.5 Load your labels

Once you have labeled records in `dev_labels.csv`, this cell reads them and joins them to the data by `record_id`. A **join** (pandas `merge`) adds the columns of one table to the rows of another table that have the same ID. After the join, the column `own_label` holds your label for the records you labeled; it is empty for all others. The cell only counts the held-out labels. They stay unused until MS3.

In [ ]:
if OWN_LABEL and DEV_LABELS.exists():
    # Read the IDs as text, so the join matches them exactly (an ID such as 00123 keeps its zeros).
    dev_labels = pd.read_csv(DEV_LABELS, dtype={"record_id": str})
    # Keep the ID and the label, and drop the rows you have not labeled yet.
    dev_labels = dev_labels[["record_id", "own_label"]].dropna(subset=["own_label"])
    # merge joins the labels to data by ID; how="left" keeps every record of data, labeled or not.
    # drop(...) first removes an own_label column from an earlier run of this cell.
    data = data.drop(columns="own_label", errors="ignore").merge(dev_labels, on="record_id", how="left")
    labeled = data[data["own_label"].notna()]
    print("labeled development records:", len(labeled), "| outside train (should be 0):", int(labeled["split"].ne("train").sum()))
    print(labeled["own_label"].value_counts().to_string())
elif OWN_LABEL:
    print("No dev_labels.csv yet. Copy the template, label some records, then run the notebook again.")
if OWN_LABEL and HELDOUT_LABELS.exists():
    heldout_labels = pd.read_csv(HELDOUT_LABELS, dtype={"record_id": str})
    # The IDs in your held-out file are protected too, even if you later change the sample size.
    protected_ids.update(heldout_labels["record_id"])
    n_done = int(heldout_labels["own_label"].notna().sum())
    print(f"held-out labels filled in: {n_done} of {len(heldout_labels)} (finish them by the MS2 freeze)")

**Your turn B (own label only).** Label 20 records of the development sample with your draft guide. Read each text in full in the file or with `show_examples(..., chars=None)`. Note every case where the guide did not tell you what to do, then revise the guide. If your target is a column of the data, skip this.

In [ ]:
# Your turn B
# TODO 1: copy dev_labels_template.csv to dev_labels.csv (in project/data/private/annotations/).
# TODO 2: fill the own_label column for 20 records, and use the note column for doubts.
# TODO 3: run the notebook again; Part 5.5 then counts your labels.
# TODO 4: write the guide changes in the cell below and in project/docs/annotation_guide.md.

**Your notes (Your turn B)**

- Records labeled (IDs):
- Cases the guide did not cover (ID, what was unclear):
- Changes to the guide:

## Part 6 Write the evaluation plan now

Decide how you will judge success before you see any result. Then nobody, including you, can move the target after the fact. Fill in the plan. The cell flags any field that is still empty. The MS1 brief has a metric menu: a main metric and a baseline for each type of task, and the module that teaches it. The main metric and the success threshold are **provisional until MS2**: you may change them once, with a reason, when you freeze the final test plan in MS2. You freeze the selected method before you score it in MS3.

The two data fields are filled in from your label route, so the saved plan always matches what the notebook does. Do not type over them.

In [ ]:
# The data roles follow your label route.
if OWN_LABEL:
    data_for_tuning = "own label: the labeled development sample from train only; valid is not used for tuning"
    data_for_final_check = (f"own label: the held-out sample from valid ({len(heldout_ids)} records), "
                            "labeled before the MS2 freeze and scored once in MS3")
else:
    data_for_tuning = "train to fit, valid to compare and tune"
    data_for_final_check = "test, scored once in MS3, after the plan was frozen in MS2 and the method before scoring"

EVALUATION_PLAN = {
    "task": "",               # for example: flag reviews that mention a delivery problem
    "output": "",             # what the decision owner receives
    "target": LABEL_COL,
    "baseline": "",           # the simple method you must beat, for example: majority class, then TF-IDF + logistic regression
    "candidate_methods": "",  # one or two methods you may compare in MS3
    "main_metric": "",        # see the metric menu in the MS1 brief
    "success_criterion": "",  # the business threshold (provisional until MS2)
    "data_for_tuning": data_for_tuning,
    "data_for_final_check": data_for_final_check,
    "uncertainty": "",        # for example: k of n now; a paired comparison with its bootstrap interval in MS3 (Lab 13)
}
# One row per field of the plan: ok, or CHECK when it is still empty.
check_rows = []
for field, value in EVALUATION_PLAN.items():
    if value.strip():
        status = "ok"
    else:
        status = "CHECK: empty"
    check_rows.append({"field": field, "status": status})
plan_check = pd.DataFrame(check_rows)
(OUT / "evaluation_plan.json").write_text(json.dumps(EVALUATION_PLAN, indent=2))
plan_check

## Part 7 A first baseline (for MS2)

You do not need a model for MS1, and classification is taught in Module 8. Keep `RUN_MS2_BASELINE = False` until then. For MS2, set it to `True` in Part 1.3. This part then shows whether your target can be predicted from the text.

- With a **column of the data** as the target, both methods are fitted on `train` and scored on `valid`.
- With **your own label**, they are fitted on 70% of your labeled development records and checked on the other 30%. Related records can fall in both parts, so read this score as a rough first check. Your held-out sample and `test` are never used.
- The **majority baseline** always guesses the most common training value.
- The **TF-IDF + logistic regression** pipeline is the course's standard simple model (Lab 8). It reads your text column.

Records without a value for the target are left out, and the cell says how many. If the target cannot be used yet, for example because it has only one value, the cell skips the model and says why. **Macro F1** averages the F1 score of each class, so small classes count as much as large ones. It runs from 0 to 1; higher is better.

For ratings (1 to 5 stars), consider grouping them first, for example 1–2, 3 and 4–5, and say why in your plan.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

baseline_scores = None
skip = ""
if not RUN_MS2_BASELINE:
    skip = "Part 7 is for MS2. Set RUN_MS2_BASELINE = True in Part 1.3 when you get there."
elif OWN_LABEL and LABEL_COL not in data.columns:
    skip = "No labels loaded yet. Label at least 20 development records first (see Part 5.5)."
elif LABEL_COL not in data.columns:
    skip = f"'{LABEL_COL}' is not a column. Check LABEL_COL in Part 1.3, or load your labels in Part 5.5."
else:
    # Only records with a value for the target can be used to fit or to score.
    has_target = data[LABEL_COL].notna()
    train_with_target = data[has_target & data["split"].eq("train")]
    if not OWN_LABEL:
        print("records left out because the target is empty:", int((~has_target).sum()))
    if OWN_LABEL and len(train_with_target) < 20:
        skip = "Label at least 20 development records first."
    elif OWN_LABEL:
        # train_test_split shuffles the labeled records with the seed and keeps 30% apart for the check.
        fit_rows, check_rows = train_test_split(train_with_target, test_size=0.3, random_state=SEED)
        check_name = "development check (30% of your labels)"
    else:
        fit_rows = train_with_target
        check_rows = data[has_target & data["split"].eq("valid")]
        check_name = "valid"
    if not skip and fit_rows[LABEL_COL].nunique() < 2:
        skip = "The fitting records have only one target value, so there is nothing to learn. Choose another target or subset."

if skip:
    print("Skipped:", skip)
else:
    check_protected(fit_rows)
    check_protected(check_rows)
    # The labels as text, so that number and word labels are handled the same way.
    y_fit = fit_rows[LABEL_COL].astype(str)
    y_check = check_rows[LABEL_COL].astype(str)
    # The majority baseline: the most common fitting label (mode()[0] takes the first one if tied).
    majority = y_fit.mode()[0]
    # Keep words and two-word phrases found in at least 5 texts (at least 2 with a small own-label sample).
    min_texts = 5
    if OWN_LABEL:
        min_texts = 2
    # make_pipeline chains TF-IDF and logistic regression; max_iter=1000 gives the solver enough steps to finish.
    pipeline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=min_texts),
                             LogisticRegression(max_iter=1000))
    # fit learns from the fitting records only; predict labels the check records.
    pipeline.fit(fit_rows["text"], y_fit)
    check_pred = pipeline.predict(check_rows["text"])
    # [majority] * len(check_rows) repeats the majority label once for every check record.
    baseline_scores = pd.DataFrame([
        {"method": f"majority ({majority})", "macro F1": f1_score(y_check, [majority] * len(check_rows), average="macro")},
        {"method": "TF-IDF + logistic regression", "macro F1": f1_score(y_check, check_pred, average="macro")},
    ]).round(3)
    baseline_scores["checked on"] = check_name
    baseline_scores["n checked"] = len(check_rows)
    baseline_scores.to_csv(OUT / "first_baseline.csv", index=False)
    # Record-level predictions for MS2: one row per checked record, with its ID, true label and both predictions.
    predictions = pd.DataFrame({"record_id": check_rows["record_id"], "true_label": y_check,
                                "majority": majority, "tfidf_logreg": check_pred})
    # predict_proba gives one probability per class; max(axis=1) keeps the probability of the predicted class.
    predictions["tfidf_logreg_prob"] = pipeline.predict_proba(check_rows["text"]).max(axis=1).round(3)
    predictions.to_csv(OUT / "first_baseline_predictions.csv", index=False)
    print(baseline_scores.to_string(index=False), end="\n\n")
    # Precision and recall of each class for the TF-IDF model; recall is the share of a class that it finds.
    print(classification_report(y_check, check_pred, zero_division=0))

If the TF-IDF model is clearly above the majority guess, the target is predictable from the text in this sample. That does not yet show the method is good enough for the decision. Look at the recall of each class, and read some of the costly errors, as in Lab 8. If the two methods are close, check your labels and your question before you add a more complex method.

**Your turn C (MS2, Week 9).** Part 7 gives a first version of both baselines and saves `first_baseline_predictions.csv`: one row per checked record, with its ID, the true label, both predictions and the model's probability. For MS2, finish the work in your own baseline notebook (`project/notebooks/02_baseline.ipynb`).

In [ ]:
# Your turn C (MS2)
# TODO 1: copy Part 7 to project/notebooks/02_baseline.ipynb; save the record-level predictions to project/outputs/ms2/.
# TODO 2: list the errors of the TF-IDF model: predictions[predictions["tfidf_logreg"].ne(predictions["true_label"])]
# TODO 3: review at least 10 errors (or all, if fewer) in full; add a column error_type and save the error table.
# TODO 4: count the errors per type (value_counts) and note which types cost the decision owner most.
# TODO 5: write project/docs/final_test_plan.md and commit it before any test result exists (see the MS2 brief).

**Your notes (Your turn C)**

- Errors reviewed (IDs):
- Error types and counts:
- The costliest error type, and why:
- Commit of `final_test_plan.md` (date):

## Part 8 Save the outputs

### 8.1 Save the tables

Only tables without text are saved to the outputs folder. Never commit the raw file, or text samples of data that may not be shared. Your project README explains how to download the data again and gives its hash. The cell lists every file this run wrote.

In [ ]:
# The provenance as a two-column table (.T turns the one row into a column), then the audit tables.
pd.DataFrame([provenance]).T.rename(columns={0: "value"}).to_csv(OUT / "provenance.csv")
audit.to_csv(OUT / "audit_text_checks.csv", index=False)
exclusions.to_csv(OUT / "audit_excluded_texts.csv", index=False)
pii_table.to_csv(OUT / "audit_personal_details_scan.csv", index=False)
split_table.to_csv(OUT / "split_counts.csv")
# The files written in this run, for the manifest.
saved = [OUT / "provenance.csv", OUT / "audit_text_checks.csv", OUT / "audit_excluded_texts.csv",
         OUT / "audit_personal_details_scan.csv", OUT / "split_counts.csv", SPLIT_FILE,
         OUT / "dropped_cross_split_copies.csv", OUT / "protected_test_ids.csv", OUT / "evaluation_plan.json"]
if label_counts is not None:
    label_counts.rename("records").to_csv(OUT / "audit_label_counts.csv")
    saved.append(OUT / "audit_label_counts.csv")
if baseline_scores is not None:
    saved.extend([OUT / "first_baseline.csv", OUT / "first_baseline_predictions.csv"])

### 8.2 Reproducibility record

The manifest is the record a reader uses to check a rerun. It holds your settings, the seed and the package versions. It also holds a SHA-256 hash of each input file, of your label files and of every output of this run. A **SHA-256 hash** is a 64-character code computed from a file's bytes; if one character changes, the hash changes. The **fingerprint** of an ID list is the hash of the sorted IDs; it shows that the protected test set never changed. Module 14 explains these records. You only need to run the two cells.

In [ ]:
from datetime import datetime, timezone
import hashlib
import platform
import matplotlib
import sklearn

def text_sha256(path):
    """Take a file path; return the SHA-256 hash of the file's bytes (Windows line endings read as \n)."""
    file_bytes = Path(path).read_bytes().replace(b"\r\n", b"\n")
    return hashlib.sha256(file_bytes).hexdigest()

def ids_sha256(ids):
    """Take a list of IDs; return one fingerprint of the list (the order of the IDs does not matter)."""
    id_texts = []
    for record_id in ids:
        id_texts.append(str(record_id))
    joined = "\n".join(sorted(id_texts))
    return hashlib.sha256(joined.encode()).hexdigest()

def file_record(path):
    """Take a file path; return its path inside the repository and the first 12 characters of its hash."""
    path = Path(path)
    # relative_to(REPO) drops the "../.." part; as_posix() writes the path with / on every computer.
    return {"file": path.relative_to(REPO).as_posix(), "sha256": text_sha256(path)[:12]}

versions = {"python": platform.python_version(), "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__}

# Your data file, with the start of its hash; it is committed only in demo mode (the course file).
data_hash = text_sha256(DATA_FILE)
input_files = [{"file": DATA_PATH, "sha256": data_hash[:12], "committed": DEMO}]
print("data file sha256 (for your README):", data_hash)

In [ ]:
# The label files, if you have them: their hashes show later that labels did not change after you saw results.
label_files = []
for path in [DEV_LABELS, HELDOUT_LABELS]:
    if path.exists():
        label_files.append(file_record(path))
output_records = []
for path in saved:
    output_records.append(file_record(path))
manifest = {
    "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "notebook": "self_selected_project_starter",
    "settings": SETTINGS,
    "seed": SEED,
    "versions": versions,
    "input_files": input_files,
    "protected_test": {"n_ids": len(test_ids), "ids_sha256": ids_sha256(test_ids)},
    "heldout_sample": {"split": "valid", "n_ids": len(heldout_ids), "ids_sha256": ids_sha256(heldout_ids)},
    "label_files": label_files,
    "outputs": output_records,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("protected test fingerprint:", manifest["protected_test"]["ids_sha256"][:12])
print("files saved in this run:")
for record in output_records + [file_record(OUT / "manifest.json")]:
    print(" ", record["file"])

## Commit your work

1. Save this notebook (keep the outputs).
2. In GitHub Desktop, look at the changed files. You should see this notebook and the files in `project/outputs/option5_setup/`. You should **not** see anything from `project/data/private/`.
3. Write a short commit message, such as "MS1 setup: audit and protected split", then commit and push.
4. Push before the deadline. The instructor grades what your repository holds at the deadline, so a commit that is not pushed does not count.

## Next steps

- **Before MS1:** email the dataset approval request by the end of Week 4; the instructor decides by the end of Week 5. Do not build your project on data that is not approved yet. If it is not approved by then, switch to one of Options 1 to 4.
- **MS1 (Week 6):** use the provenance record, the personal-details decisions, the audit tables, the saved split and the evaluation plan in your plan.
- **MS2 (Week 9):** set `RUN_MS2_BASELINE = True` and build your baseline from Part 7, with an error review as in Labs 8 and 9.
- **MS3 (Week 13):** compare your selected method with the baseline once: on the protected test, or, with your own label, on the held-out sample.
- **Use TF-IDF as your text representation.** The saved course embeddings and LLM outputs do not cover your data. The course environment has no embedding route for new texts: the embedding model is not installed. Adding it would break the final rerun in the course environment. TF-IDF (Labs 4, 6 and 8) is the supported representation for Option 5.
- **An LLM is optional.** If you try one on a small sample of approved, non-personal text, save each output with the model, version, date and prompt. Ask early in the course Q&A if your computer cannot run a step.